# 🧬 OncRAG — LLM-Only Baseline (Ablation Study)
### Direct Question → Mistral-7B → Answer  |  No Retrieval  |  No Context
**Purpose:** Compare against full OncRAG (BM25+FAISS+MMR+LAQ) to quantify RAG contribution.

**Files needed on /content:**
- `evaluator_baseline.py` — standalone evaluator, loads its own Mistral-7B ← upload this
- `cleaned_output.json` — 200 QA pairs (same as full eval)

> **Do NOT upload** `chain.py`, `retriever.py`, `chunker.py`, `memory.py`, `query_analyzer.py`,
> or any index files — this baseline is fully standalone and never imports any of them.


## 📦 Step 1 — Install Dependencies
Same pinned versions as full OncRAG eval.

In [ ]:
!pip install -q "transformers==4.46.3" "bitsandbytes==0.46.1" "accelerate==0.34.2"
!pip install -q sentence-transformers
!pip install -q bert-score
!pip install -q rouge_score nltk
print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.8 MB/s eta 0:00:00
✅ All packages installed


## ✅ Step 2 — Verify Package Versions

In [ ]:
import bitsandbytes, transformers
from transformers.utils import is_bitsandbytes_available
import torch

print(f"bitsandbytes : {bitsandbytes.__version__}  (need 0.46.1)")
print(f"transformers : {transformers.__version__}  (need 4.46.3)")
print(f"bnb available: {is_bitsandbytes_available()}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f"GPU          : {torch.cuda.get_device_name(0)}  ({total:.1f} GB)")

if not is_bitsandbytes_available():
    print("\n⚠️  bitsandbytes not available — re-run Step 1 then restart runtime")
else:
    print("\n✅ Ready to proceed")

bitsandbytes : 0.46.1  (need 0.46.1)
transformers : 4.46.3  (need 4.46.3)
bnb available: True
CUDA         : True
GPU          : Tesla T4  (15.6 GB)

✅ Ready to proceed


## 🖥️ Step 3 — GPU Check & Path Setup

In [ ]:
import torch, gc, os, sys

def gpu_mem(label=""):
    if torch.cuda.is_available():
        used  = torch.cuda.memory_allocated()/1e9
        total = torch.cuda.get_device_properties(0).total_memory/1e9
        tag   = f" [{label}]" if label else ""
        print(f"GPU{tag}: {used:.2f}/{total:.2f} GB  (free: {total-used:.2f} GB)")
    else:
        print("⚠️  No GPU")

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Clear stale module cache
for mod in list(sys.modules.keys()):
    if any(m in mod for m in ["chain", "evaluator_baseline"]):
        del sys.modules[mod]

os.chdir("/content")
sys.path.insert(0, "/content")

QA_JSON    = "/content/cleaned_output.json"
OUTPUT_DIR = "/content/eval_results_baseline"

gpu_mem("baseline")
print(f"\nPaths:")
print(f"  QA_JSON    : {QA_JSON}")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}")

GPU [baseline]: 0.00/15.64 GB  (free: 15.64 GB)

Paths:
  QA_JSON    : /content/cleaned_output.json
  OUTPUT_DIR : /content/eval_results_baseline


## 📁 Step 4 — Verify Required Files

In [ ]:
all_ok = True
print("── Python modules ──────────────────────────────────────")
for f in ["evaluator_baseline.py"]:
    ok   = os.path.exists(f"/content/{f}")
    size = os.path.getsize(f"/content/{f}")/1e3 if ok else 0
    print(f"  {'✅' if ok else '❌ MISSING'}  {f:<30} ({size:.0f} KB)")
    if not ok: all_ok = False

print("\n── Data files ──────────────────────────────────────────")
ok   = os.path.exists(QA_JSON)
size = os.path.getsize(QA_JSON)/1e6 if ok else 0
print(f"  {'✅' if ok else '❌ MISSING'}  cleaned_output.json          ({size:.1f} MB)")
if not ok: all_ok = False

if all_ok:
    print("\n✅ All required files present — ready to run")
else:
    print("\n❌ Upload missing files before continuing")

── Python modules ──────────────────────────────────────
  ✅  chain.py                       (12 KB)
  ✅  evaluator_baseline.py          (20 KB)

── Data files ──────────────────────────────────────────
  ✅  cleaned_output.json          (0.1 MB)

✅ All required files present — ready to run


## 🤖 Step 5 — Load Mistral-7B (4-bit NF4)
Loads via `evaluator_baseline.py`'s own standalone loader (~2–3 min). No `chain.py` involved —
this baseline never touches the RAG stack.


In [ ]:
from evaluator_baseline import _get_model_and_tok
_get_model_and_tok()   # triggers standalone 4-bit Mistral load — no chain.py needed
gpu_mem("after Mistral load")

free_gb = (
    torch.cuda.get_device_properties(0).total_memory
    - torch.cuda.memory_allocated()
) / 1e9 if torch.cuda.is_available() else 0
print(f"\n✅ Mistral loaded — {free_gb:.1f} GB VRAM free for eval")


GPU [after Mistral load]: 4.13/15.64 GB  (free: 11.51 GB)

✅ Mistral loaded — 11.5 GB VRAM free for eval


## 🔬 Step 6 — Verify Baseline Evaluator

In [ ]:
import evaluator_baseline as eb

checks = {
    "llm_only_answer function present":    hasattr(eb, "llm_only_answer"),
    "scope_judge function present":         hasattr(eb, "scope_judge"),
    "BaselineEvaluator class present":      hasattr(eb, "BaselineEvaluator"),
    "BERTScorer used":                      hasattr(eb, "bertscore"),
    "No retrieval in llm_only_answer":      "retrieve" not in eb.llm_only_answer.__code__.co_names,
}

all_good = True
for check, result in checks.items():
    flag = "✅" if result else "❌"
    print(f"  {flag}  {check}")
    if not result: all_good = False

if all_good:
    print("\n✅ Baseline evaluator verified — safe to run")
else:
    print("\n❌ Re-upload evaluator_baseline.py and re-run Steps 3+5+6")

  ✅  llm_only_answer function present
  ✅  scope_judge function present
  ✅  BaselineEvaluator class present
  ✅  BERTScorer used
  ✅  No retrieval in llm_only_answer

✅ Baseline evaluator verified — safe to run


In [ ]:
import sys
for mod in list(sys.modules.keys()):
    if "evaluator_baseline" in mod:
        del sys.modules[mod]

from evaluator_baseline import llm_only_answer

## 🧪 Step 7 — Quick Spot Check (3 questions)
Verify the baseline generates answers (not just empty strings) before committing to the full run.

In [ ]:
from evaluator_baseline import llm_only_answer

spot_qs = [
    "What are the treatment options for locally advanced head and neck cancer?",
    "How does HER2 overexpression affect breast cancer prognosis?",
    "What is the most common presenting symptom for laryngeal cancer?",
]

print("Spot check — LLM-only answers (no context):\n")
for q in spot_qs:
    ans = llm_only_answer(q)
    tag = "❌ EMPTY" if not ans or "not available" in ans.lower() else "✅ ANSWERED"
    print(f"{tag}")
    print(f"  Q: {q[:70]}")
    print(f"  A: {ans[:150]}\n")

gpu_mem("after spot check")

Spot check — LLM-only answers (no context):



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ ANSWERED
  Q: What are the treatment options for locally advanced head and neck canc
  A: Locally advanced head and neck cancer refers to tumors that have grown large or invaded nearby structures but have not spread to distant sites. Treatm

✅ ANSWERED
  Q: How does HER2 overexpression affect breast cancer prognosis?
  A: HER2 overexpression is a characteristic feature of some breast cancers, leading to an aggressive clinical course. HER2-positive breast cancers have a 

✅ ANSWERED
  Q: What is the most common presenting symptom for laryngeal cancer?
  A: The most common presenting symptom for laryngeal cancer is a hoarse voice or vocal cord paralysis. Other symptoms may include coughing up blood, pain 

GPU [after spot check]: 4.13/15.64 GB  (free: 11.50 GB)


## 📊 Step 8 — Run Full Baseline Evaluation
- `MAX_Q = 20` → quick test (~20 min on T4)
- `MAX_Q = 200` → full ablation run (~2.5 hrs on T4)

**SCOPE judging is deferred** (`live_scope_judge=False`, the default) — this pass only
generates answers with Mistral and computes ROUGE/BLEU/BERTScore/faithfulness. SCOPE
fields are left `pending`. This is deliberate: loading the 8B judge model alongside
Mistral on the same GPU is what causes the "dispatched on CPU or disk" crash. After
this cell finishes, we unload Mistral, then Step 8b loads only the judge model and
fills SCOPE in — that's the fix.

Auto-checkpoints after every question — safe to interrupt and resume via Step 9.


In [ ]:
# Flush module cache
import sys
for mod in list(sys.modules.keys()):
    if "evaluator_baseline" in mod:
        del sys.modules[mod]

In [ ]:
# Confirm the fix is live
import evaluator_baseline as eb
import json
qs = json.loads(open("/content/cleaned_output.json").read())
item = qs[0]
q = item.get("q", item.get("question", ""))
a = item.get("a", item.get("answer", ""))
print(f"Q: {q}")
print(f"A: {a}")

Q: What are the three main anatomical divisions of the larynx?
A: The larynx is anatomically divided into the supraglottic larynx, the glottis, and the subglottis .


In [ ]:
MAX_Q = 200  # ← set to 20 for a quick test first

# Clear old checkpoint for a fresh run
!rm -f /content/eval_results_baseline/checkpoint_baseline.json
!rm -f /content/eval_results_baseline/results_baseline_*.json
!rm -f /content/eval_results_baseline/report_baseline_*.json
!rm -f /content/eval_results_baseline/report_baseline_*.txt

# Reload fresh
for mod in list(sys.modules.keys()):
    if "evaluator_baseline" in mod:
        del sys.modules[mod]

from evaluator_baseline import BaselineEvaluator

ev = BaselineEvaluator(
    questions_json = QA_JSON,
    output_dir     = OUTPUT_DIR,
    max_questions  = MAX_Q,
)
report = ev.run()
# Free Mistral's VRAM now that generation is done — Step 8b needs
# that headroom to load the judge model.
from evaluator_baseline import unload_generator_model
unload_generator_model()


Baseline eval — 200 questions (0 already done, 200 remaining)

[  1/200] Q001  (simple/diagnosis)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


       BS=0.135  faith=0.078  SCOPE=3.00/5
[  2/200] Q002  (moderate/treatment)
       BS=0.143  faith=0.120  SCOPE=3.00/5
[  3/200] Q003  (complex/treatment)
       BS=-0.010  faith=0.127  SCOPE=4.80/5
[  4/200] Q004  (simple/diagnosis)
       BS=0.034  faith=0.174  SCOPE=3.00/5
[  5/200] Q005  (moderate/epidemiology)
       BS=0.233  faith=0.151  SCOPE=4.90/5
[  6/200] Q006  (moderate/staging)
       BS=0.095  faith=0.115  SCOPE=4.80/5
[  7/200] Q007  (complex/biomarker)
       BS=0.231  faith=0.264  SCOPE=4.60/5
[  8/200] Q008  (simple/diagnosis)
       BS=0.422  faith=0.226  SCOPE=3.00/5
[  9/200] Q009  (moderate/prognosis)
       BS=-0.023  faith=0.124  SCOPE=4.80/5
[ 10/200] Q010  (simple/treatment)
       BS=0.244  faith=0.174  SCOPE=3.00/5
[ 11/200] Q011  (complex/diagnosis)
       BS=0.124  faith=0.120  SCOPE=3.00/5
[ 12/200] Q012  (moderate/diagnosis)
       BS=0.194  faith=0.143  SCOPE=3.00/5
[ 13/200] Q013  (simple/epidemiology)
       BS=0.018  faith=0.094  SCOPE=4.60/5
[ 

## 🧑‍⚖️ Step 8b (optional) — Re-judge with independent judge model
Run this **after** Step 8. Step 8 already unloaded Mistral at the end, so there's
free VRAM here for `Llama3-Med42-8B` — the two models are never on the GPU together.
Re-scores existing answers with `Llama3-Med42-8B` instead of Mistral judging itself,
then recomputes the aggregate report with real SCOPE numbers.


In [5]:
from huggingface_hub import login
login()

In [1]:
!pip install -q "transformers==4.46.3" "bitsandbytes==0.46.1" "accelerate==0.34.2" huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 105.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
# ── Step 8b.1 — Re-judge existing baseline answers with Llama3-Med42-8B ───────
# Run this AFTER Step 8 (full run) and the huggingface login cell above.
# Does NOT reload Mistral — only the separate judge model.

import json, time
from evaluator_baseline import scope_judge, BaselineEvaluator

import glob
_candidates = sorted(glob.glob("/content/eval_results_baseline/results_baseline_*.json"))
if not _candidates:
    raise FileNotFoundError(
        "No results_baseline_*.json found in /content/eval_results_baseline — "
        "run Step 8 (the full evaluation) first."
    )
INPUT_FILE  = _candidates[-1]   # most recent run
OUTPUT_FILE = "/content/eval_results_baseline/results_baseline_med42judge.json"
print(f"Re-judging from: {INPUT_FILE}")

with open(INPUT_FILE) as f:
    results = json.load(f)

SCOPE_WEIGHTS = BaselineEvaluator.SCOPE_WEIGHTS
print(f"Re-judging {len(results)} questions with Llama3-Med42-8B...\n")

t0 = time.time()
for i, row in enumerate(results, 1):
    sc = scope_judge(row["question"], row["reference"], row["answer"])

    old_weighted = row["scope_weighted"]
    row["scope_S"]   = sc["scope_S"]
    row["scope_C"]   = sc["scope_C"]
    row["scope_O"]   = sc["scope_O"]
    row["scope_P"]   = sc["scope_P"]
    row["scope_E"]   = sc["scope_E"]
    row["scope_Emp"] = sc["scope_Emp"]
    row["scope_weighted"] = sum(SCOPE_WEIGHTS[k] * sc[k] for k in SCOPE_WEIGHTS)
    row["scope_judge_model"] = "Llama3-Med42-8B"   # so you can tell old vs new rows apart

    print(f"[{i:>3}/{len(results)}] {row['qid']}  "
          f"old={old_weighted:.2f} -> new={row['scope_weighted']:.2f}/5")

    # checkpoint every 20 questions in case Colab disconnects
    if i % 20 == 0 or i == len(results):
        with open(OUTPUT_FILE, "w") as f:
            json.dump(results, f, indent=2)

elapsed = time.time() - t0
print(f"\n✅ Done in {elapsed/60:.1f} min. Saved -> {OUTPUT_FILE}")


# ── Step 8b.2 — Recompute the aggregate report with the new scores ────────────
from statistics import mean, stdev

def avg(key):
    return mean(r[key] for r in results)

scope_keys = ["scope_S", "scope_C", "scope_O", "scope_P", "scope_E", "scope_Emp"]
weighted_vals = [r["scope_weighted"] for r in results]

new_report = {
    "mode": "llm_only_baseline_med42judge",
    "n_questions": len(results),
    "not_found_count": sum(1 for r in results if r.get("not_found")),
    "lexical": {
        "rouge1": avg("rouge1"), "rouge2": avg("rouge2"),
        "rougeL": avg("rougeL"), "bleu1": avg("bleu1"),
    },
    "semantic": {"bertscore_f1": avg("bertscore_f1")},
    "faithfulness": {
        "faithfulness": avg("faithfulness"),
        "answer_relevance": avg("answer_relevance"),
    },
    "scope": {
        "S_safety": avg("scope_S"), "C_completeness": avg("scope_C"),
        "O_originality": avg("scope_O"), "P_precision": avg("scope_P"),
        "E_efficiency": avg("scope_E"), "Emp_empathy": avg("scope_Emp"),
        "weighted_total": mean(weighted_vals),
        "std": stdev(weighted_vals) if len(weighted_vals) > 1 else 0.0,
    },
}

print("\n" + "=" * 60)
print("  BASELINE — Med42-8B-JUDGED S.C.O.P.E.E RESULTS")
print("=" * 60)
for key, label in [
    ("S_safety", "S  Safety      "), ("C_completeness", "C  Completeness"),
    ("O_originality", "O  Originality "), ("P_precision", "P  Precision   "),
    ("E_efficiency", "E  Efficiency  "), ("Emp_empathy", "E  Empathy     "),
]:
    print(f"  {label}: {new_report['scope'][key]:.2f}/5.00")
print(f"  Weighted Total    : {new_report['scope']['weighted_total']:.2f}/5.00 "
      f"(std={new_report['scope']['std']:.2f})")
print("=" * 60)

with open("/content/eval_results_baseline/report_baseline_med42judge.json", "w") as f:
    json.dump(new_report, f, indent=2)
print("\n✅ Report saved -> /content/eval_results_baseline/report_baseline_med42judge.json")

Re-judging 200 questions with Llama3-Med42-8B...

[Judge] Loading independent judge model: m42-health/Llama3-Med42-8B ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

[Judge] ✅ Llama3-Med42-8B loaded (separate from the Mistral generator)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[  1/200] Q001  old=3.00 -> new=4.90/5
[  2/200] Q002  old=3.00 -> new=4.90/5
[  3/200] Q003  old=4.80 -> new=4.50/5
[  4/200] Q004  old=3.00 -> new=3.00/5
[  5/200] Q005  old=4.90 -> new=4.90/5
[  6/200] Q006  old=4.80 -> new=4.90/5
[  7/200] Q007  old=4.60 -> new=4.90/5
[  8/200] Q008  old=3.00 -> new=4.50/5
[  9/200] Q009  old=4.80 -> new=4.50/5
[ 10/200] Q010  old=3.00 -> new=3.00/5
[ 11/200] Q011  old=3.00 -> new=4.90/5
[ 12/200] Q012  old=3.00 -> new=4.90/5
[ 13/200] Q013  old=4.60 -> new=3.00/5
[ 14/200] Q014  old=3.00 -> new=4.50/5
[ 15/200] Q015  old=4.80 -> new=4.90/5
[ 16/200] Q016  old=4.80 -> new=3.00/5
[ 17/200] Q017  old=3.00 -> new=4.90/5
[ 18/200] Q018  old=3.00 -> new=4.90/5
[ 19/200] Q019  old=3.00 -> new=3.00/5
[ 20/200] Q020  old=3.00 -> new=4.40/5
[ 21/200] Q021  old=4.80 -> new=4.50/5
[ 22/200] Q022  old=3.00 -> new=5.00/5
[ 23/200] Q023  old=4.80 -> new=4.50/5
[ 24/200] Q024  old=4.80 -> new=4.90/5
[ 25/200] Q025  old=4.80 -> new=4.50/5
[ 26/200] Q026  old=4.80 

In [8]:
# ── Step 8b.3 — Download results to your own machine ─────────────────────────
from google.colab import files

files.download(OUTPUT_FILE)                                                    # results_baseline_med42judge.json
files.download("/content/eval_results_baseline/report_baseline_med42judge.json")  # aggregate report

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🔄 Step 9 — Resume Interrupted Evaluation
Use if eval stopped mid-way. **Do NOT clear the checkpoint.**

In [ ]:
import json as _j

ckpt = "/content/eval_results_baseline/checkpoint_baseline.json"
if os.path.exists(ckpt):
    done = _j.loads(open(ckpt).read())
    print(f"✅ Checkpoint: {len(done)} done, {200-len(done)} remaining")
else:
    print("No checkpoint — will start from Q001")

used = torch.cuda.memory_allocated()/1e9 if torch.cuda.is_available() else 0
if used < 2.0:
    print("⚠️  Model not loaded — run Steps 1, 3, 5 first, then come back here")
else:
    print(f"✅ Mistral still in GPU ({used:.1f} GB) — safe to resume")

for mod in list(sys.modules.keys()):
    if "evaluator_baseline" in mod:
        del sys.modules[mod]

from evaluator_baseline import BaselineEvaluator

ev = BaselineEvaluator(
    questions_json = QA_JSON,
    output_dir     = OUTPUT_DIR,
    max_questions  = 200,
)
report = ev.run()

## 📈 Step 10 — Results Dashboard
Shows the same metric layout as the full OncRAG dashboard for direct side-by-side comparison.
Retrieval metrics will all read 0.0 — that is expected and correct for an LLM-only baseline.

In [7]:
def dashboard_baseline(r, results):
    sep = "=" * 70
    nf  = r.get("not_found_count", "?")
    pct = int(nf/r['n_questions']*100) if isinstance(nf,int) else "?"
    print(f"\n{sep}")
    print("  LLM-ONLY BASELINE — EVALUATION DASHBOARD")
    print(f"  {r['n_questions']} questions  |  Not-found: {nf} ({pct}%)  |  Mode: no retrieval")
    print(sep)

    print("\n  ── RETRIEVAL  (all 0.0 — no retrieval in baseline) ──────────")
    for name in ["Precision@5","Recall@5","MRR","NDCG@5","Hit-Rate@5"]:
        print(f"     {name:<18}: 0.0000")

    lex = r["lexical"]
    print("\n  ── GENERATION — Lexical ─────────────────────────────────────")
    for k, v in lex.items():
        print(f"     {k:<20}: {v:.4f}")

    print("\n  ── GENERATION — Semantic ────────────────────────────────────")
    print(f"     BERTScore F1        : {r['semantic']['bertscore_f1']:.4f}")

    faith = r["faithfulness"]
    print("\n  ── FAITHFULNESS & RELEVANCE ─────────────────────────────────")
    for k, v in faith.items():
        print(f"     {k:<26}: {v:.4f}")

    sc = r["scope"]
    print("\n  ── S.C.O.P.E.E  LLM-as-judge (/5.0) ───────────────────────")
    for key, label in [
        ("S_safety",       "S  Safety      "),
        ("C_completeness", "C  Completeness"),
        ("O_originality",  "O  Originality "),
        ("P_precision",    "P  Precision   "),
        ("E_efficiency",   "E  Efficiency  "),
        ("Emp_empathy",    "E  Empathy     "),
    ]:
        v    = sc.get(key, "N/A")
        flag = "✅" if isinstance(v,(int,float)) and v >= 3.5 else "⚠️ "
        print(f"  {flag} {label}: {v}")
    wt   = sc["weighted_total"]
    flag = "✅" if wt >= 3.5 else "⚠️ "
    print(f"  {flag} {'Weighted Total':<18}: {wt:.2f}/5.00  (std={sc['std']:.2f})")
    print(sep)

    if results:
        from collections import defaultdict
        print("\n  ── Per-Difficulty Breakdown ─────────────────────────────────")
        by_diff = defaultdict(list)
        for row in results: by_diff[row["difficulty"]].append(row)
        for diff in ["simple","moderate","complex"]:
            rows = by_diff.get(diff, [])
            if not rows: continue
            sc_a = sum((r["scope_S"]+r["scope_C"]+r["scope_O"]+
                        r["scope_P"]+r["scope_E"]+r.get("scope_Emp",3))/6
                       for r in rows)/len(rows)
            nf_c = sum(1 for r in rows if r.get("not_found",False))
            print(f"     {diff:<10} n={len(rows):<4} SCOPE={sc_a:.2f}/5  not_found={nf_c}")

        print("\n  ── Per-Category Breakdown ───────────────────────────────────")
        by_cat = defaultdict(list)
        for row in results: by_cat[row["category"]].append(row)
        for cat in sorted(by_cat):
            rows = by_cat[cat]
            sc_a = sum((r["scope_S"]+r["scope_C"]+r["scope_O"]+
                        r["scope_P"]+r["scope_E"]+r.get("scope_Emp",3))/6
                       for r in rows)/len(rows)
            emp  = sum(r.get("scope_Emp",3) for r in rows)/len(rows)
            print(f"     {cat:<22} n={len(rows):<4} SCOPE={sc_a:.2f}/5  Empathy={emp:.2f}")

import glob as _g, json as _j
result_files = sorted(_g.glob("/content/eval_results_baseline/results_baseline_*.json"))
if result_files:
    latest  = _j.loads(open(result_files[-1]).read())
    dashboard_baseline(report, latest)
else:
    print("No results yet — run Step 8 first")

No results yet — run Step 8 first


## 📊 Step 11 — Side-by-Side Comparison Table
Paste your full OncRAG scores into `RAG_SCORES` below and run this cell to print a clean comparison.

In [ ]:
#  OncRAG scores here ──────────────────────
RAG_SCORES = {
    "rouge1":         0.0,   # ← replace with actual
    "rouge2":         0.0,
    "rougeL":         0.0,
    "bleu1":          0.0,
    "bertscore_f1":   0.0,
    "faithfulness":   0.0,
    "scope_weighted": 0.0,
    "precision_5":    0.0,
}
# ─────────────────────────────────────────────────────────────

BASE_SCORES = {
    "rouge1":         report["lexical"]["rouge1"],
    "rouge2":         report["lexical"]["rouge2"],
    "rougeL":         report["lexical"]["rougeL"],
    "bleu1":          report["lexical"]["bleu1"],
    "bertscore_f1":   report["semantic"]["bertscore_f1"],
    "faithfulness":   report["faithfulness"]["faithfulness"],
    "scope_weighted": report["scope"]["weighted_total"],
    "precision_5":    0.0,  # always 0 for baseline
}

print(f"{'Metric':<22} {'OncRAG (full)':>15} {'LLM-only':>12} {'Delta':>10}")
print("-" * 62)
for k in RAG_SCORES:
    rag  = RAG_SCORES[k]
    base = BASE_SCORES[k]
    delta = rag - base
    arrow = "▲" if delta > 0 else ("▼" if delta < 0 else "=")
    print(f"  {k:<20} {rag:>15.4f} {base:>12.4f} {arrow}{abs(delta):>8.4f}")

## 💾 Step 12 — Download Results

In [ ]:
import glob as _g
from google.colab import files as _f

result_files = sorted(
    _g.glob("/content/eval_results_baseline/*.json") +
    _g.glob("/content/eval_results_baseline/*.txt")
)

if result_files:
    print(f"Downloading {len(result_files)} files:")
    for f in result_files:
        size = os.path.getsize(f)/1e3
        print(f"  {os.path.basename(f)}  ({size:.0f} KB)")
        _f.download(f)
else:
    print("No result files yet — run Step 8 first")

  checkpoint_baseline.json  (373 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  report_baseline_20260627_095234.json  (1 KB)


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  results_baseline_20260627_095234.json  (371 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>